# Dataset repair: `coffee_rice_v001` -> `coffee_rice_v002`

Self-contained repair pipeline (no project imports, Kaggle-ready).

It fixes the five defects measured on `coffee_rice_v001`:

| # | Defect (measured on v001) | Repair |
|---|---|---|
| 1 | Coffee: 3798 images are 3 flip/rot copies of 1255 source photos; 92.5% of test images have a twin in train | keep one canonical variant per source, split by source group |
| 2 | Coffee: COCO multipolygons (65.3% of annotations, up to 214 rings) expand to 30791 YOLO instances, 86.9% of them specks < 0.1% image area | one instance = largest ring per annotation (mean area loss 0.5%), specks dropped |
| 3 | Rice: 369 annotations have empty `segmentation` plus a box covering the whole image | removed from the segmentation dataset and logged |
| 4 | Rice: capture-session shortcut (class distribution is session-dependent) and photo bursts (median inter-frame gap 4s) were split randomly | group by capture burst + verified duplicates, split by group |
| 5 | `Healthy` used as a detection category (1516 instances, median box 68% of the image) | `Healthy` becomes an image-level label; those images become background (negative) images |

Outputs go to `<OUTPUT_ROOT>/coffee_rice_v002/`. Images are **not** copied: manifests reference the
v001 image paths and `dataset_manifest.json` records `images_source_version`.

## 0. Configuration

In [1]:
from __future__ import annotations

import json, os, platform, random, re, shutil, subprocess, time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)

SOURCE_VERSION = os.environ.get("SOURCE_DATASET_VERSION", "coffee_rice_v001")
TARGET_VERSION = os.environ.get("TARGET_DATASET_VERSION", "coffee_rice_v002")
DOMAINS = ("rice", "coffee")

# Class policy: `Healthy` is an image-level label, never an instance class.
IMAGE_LEVEL_LABELS = {"Healthy"}
DETECTION_CLASSES = {
    "rice": ["BrownSpot", "Hispa", "LeafBlast"],
    "coffee": ["LeafMiner", "PowderyMildew", "Rust", "AlgalLeafSpot"],
}

# Repair thresholds
MIN_RING_POINTS = 3           # a polygon ring needs >= 3 points
MIN_AREA_FRAC = 5e-4          # drop instances smaller than 0.05% of the image
MAX_AREA_FRAC = 0.90          # a mask covering ~the whole image is a degenerate label
BURST_GAP_SECONDS = 10.0      # rice: frames within 10s belong to the same leaf burst
DUP_PIXEL_TOL = 6.0           # mean abs diff (0-255) on 64x64 gray, flip/rot invariant
PHASH_CANDIDATE_BITS = 60     # 1024-bit pHash hamming pre-filter
SPLIT_RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}
ENABLE_DUP_SCAN = os.environ.get("ENABLE_DUP_SCAN", "1") == "1"
POPULATE_IMAGES = os.environ.get("POPULATE_IMAGES", "1") == "1"
IMAGE_LINK_MODE = os.environ.get("IMAGE_LINK_MODE", "auto")


def find_source_root() -> Path:
    candidates = []
    for key in ("CLEAN_DATASET_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(key)
        if value:
            candidates += [Path(value), Path(value) / "data" / "clean" / SOURCE_VERSION]
    candidates += [
        Path("/kaggle/input/coffee-and-rice-leaf-disease-clean-dataset") / SOURCE_VERSION,
        Path("/kaggle/input/coffee-and-rice-leaf-disease-clean-dataset"),
    ]
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidates.append(parent / "data" / "clean" / SOURCE_VERSION)
    for candidate in candidates:
        if (candidate / "rice").is_dir() and (candidate / "coffee").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f"Clean dataset {SOURCE_VERSION} not found; tried {[str(c) for c in candidates]}")


SOURCE_ROOT = find_source_root()
PROJECT_ROOT = SOURCE_ROOT.parents[2] if SOURCE_ROOT.parts[-3:-1] == ("data", "clean") else Path.cwd()
DEFAULT_OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else (PROJECT_ROOT / "data" / "clean")
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", str(DEFAULT_OUTPUT))) / TARGET_VERSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR = OUTPUT_ROOT / "reports"; REPORT_DIR.mkdir(exist_ok=True)

print("source:", SOURCE_ROOT)
print("output:", OUTPUT_ROOT)

source: /Users/tuananhduong/Projects/coffee-rice-disease-detector/data/clean/coffee_rice_v001
output: /Users/tuananhduong/Projects/coffee-rice-disease-detector/data/clean/coffee_rice_v002


## 1. Load v001 manifests and COCO annotations

In [2]:
def load_domain(domain: str) -> tuple[pd.DataFrame, dict]:
    root = SOURCE_ROOT / domain
    images = pd.read_csv(root / "manifests" / "images.csv")
    with (root / "annotations" / "instances.coco.json").open("r", encoding="utf-8") as handle:
        coco = json.load(handle)
    return images, coco


def resolve_image_path(domain: str, coco_file_name: str) -> Path:
    return (SOURCE_ROOT / domain / str(coco_file_name).replace("\\", "/")).resolve()


SOURCE = {}
for domain in DOMAINS:
    images, coco = load_domain(domain)
    SOURCE[domain] = {"images": images, "coco": coco, "n_images_source": int(len(images))}
    missing = sum(0 if resolve_image_path(domain, f).exists() else 1 for f in images["coco_file_name"])
    print(f"{domain:7s} images={len(images):5d} anns={len(coco['annotations']):5d} missing_files={missing}")

rice    images= 3351 anns= 3646 missing_files=0


coffee  images= 3798 anns= 3774 missing_files=0


## 2. Annotation repair

One instance == one annotation == one outer ring. This is the defect that corrupted the Coffee
finetune: exporting every COCO ring as its own YOLO instance turned 3774 leaf annotations into
30791 instances, 86.9% of which are sub-pixel specks.

In [3]:
def ring_area(ring: list[float]) -> float:
    xs = np.asarray(ring[0::2], dtype=np.float64)
    ys = np.asarray(ring[1::2], dtype=np.float64)
    return float(abs(np.dot(xs, np.roll(ys, -1)) - np.dot(ys, np.roll(xs, -1))) / 2.0)


def clip_ring(ring: list[float], width: int, height: int) -> list[float]:
    xs = np.clip(np.asarray(ring[0::2], dtype=np.float64), 0.0, width - 1.0)
    ys = np.clip(np.asarray(ring[1::2], dtype=np.float64), 0.0, height - 1.0)
    return np.stack([xs, ys], axis=1).ravel().tolist()


def repair_annotation(ann: dict, width: int, height: int) -> tuple[dict | None, dict]:
    """Return (repaired annotation or None, ledger row)."""
    image_area = float(width * height)
    raw = ann.get("segmentation") or []
    rings = [r for r in raw if isinstance(r, list) and len(r) >= 2 * MIN_RING_POINTS]
    ledger = {
        "annotation_id": ann.get("id"),
        "image_id": ann.get("image_id"),
        "category_id": ann.get("category_id"),
        "n_rings_in": len(raw),
        "n_rings_valid": len(rings),
        "action": "kept",
        "reason": "",
        "area_frac": None,
        "fragment_area_lost": 0.0,
    }
    if not rings:
        ledger.update(action="dropped", reason="no_valid_polygon_ring")
        return None, ledger

    clipped = [clip_ring(r, width, height) for r in rings]
    areas = [ring_area(r) for r in clipped]
    keep = int(np.argmax(areas))
    total = float(sum(areas)) or 1.0
    ring, area = clipped[keep], areas[keep]
    ledger["fragment_area_lost"] = round(1.0 - area / total, 6)
    frac = area / image_area
    ledger["area_frac"] = round(frac, 8)

    if frac < MIN_AREA_FRAC:
        ledger.update(action="dropped", reason="area_below_min")
        return None, ledger
    if frac > MAX_AREA_FRAC:
        ledger.update(action="dropped", reason="area_above_max")
        return None, ledger

    xs, ys = np.asarray(ring[0::2]), np.asarray(ring[1::2])
    repaired = dict(ann)
    repaired["segmentation"] = [[round(float(v), 2) for v in ring]]
    repaired["bbox"] = [round(float(xs.min()), 2), round(float(ys.min()), 2),
                        round(float(xs.max() - xs.min()), 2), round(float(ys.max() - ys.min()), 2)]
    repaired["area"] = round(area, 2)
    repaired["iscrowd"] = 0
    if len(rings) > 1:
        ledger.update(action="kept_largest_ring", reason=f"dropped_{len(rings) - 1}_fragment_rings")
    return repaired, ledger


for domain in DOMAINS:
    coco = SOURCE[domain]["coco"]
    meta = {int(img["id"]): img for img in coco["images"]}
    kept, ledger_rows = [], []
    for ann in coco["annotations"]:
        img = meta[int(ann["image_id"])]
        repaired, ledger = repair_annotation(ann, int(img["width"]), int(img["height"]))
        ledger["file_name"] = img["file_name"]
        ledger_rows.append(ledger)
        if repaired is not None:
            kept.append(repaired)
    ledger_df = pd.DataFrame(ledger_rows)
    SOURCE[domain]["repaired_annotations"] = kept
    SOURCE[domain]["ledger"] = ledger_df
    print(f"=== {domain}: {len(coco['annotations'])} -> {len(kept)} annotations")
    print(ledger_df.groupby(["action", "reason"]).size().to_string())
    print("fragment area lost: mean={:.4f} p99={:.4f}".format(
        ledger_df["fragment_area_lost"].mean(), ledger_df["fragment_area_lost"].quantile(0.99)))

=== rice: 3646 -> 3276 annotations
action             reason                   
dropped            area_below_min                  1
                   no_valid_polygon_ring         369
kept                                            3043
kept_largest_ring  dropped_12_fragment_rings       1
                   dropped_13_fragment_rings       1
                   dropped_1_fragment_rings      161
                   dropped_2_fragment_rings       41
                   dropped_32_fragment_rings       1
                   dropped_3_fragment_rings        7
                   dropped_4_fragment_rings        4
                   dropped_5_fragment_rings        9
                   dropped_6_fragment_rings        3
                   dropped_7_fragment_rings        2
                   dropped_8_fragment_rings        3
fragment area lost: mean=0.0057 p99=0.2747


=== coffee: 3774 -> 3716 annotations
action             reason                    
dropped            area_below_min                  58
kept                                             1254
kept_largest_ring  dropped_100_fragment_rings       1
                   dropped_106_fragment_rings       1
                   dropped_109_fragment_rings       1
                   dropped_10_fragment_rings       55
                   dropped_111_fragment_rings       1
                   dropped_113_fragment_rings       1
                   dropped_11_fragment_rings       46
                   dropped_12_fragment_rings       40
                   dropped_132_fragment_rings       1
                   dropped_13_fragment_rings       45
                   dropped_143_fragment_rings       1
                   dropped_14_fragment_rings       44
                   dropped_15_fragment_rings       32
                   dropped_16_fragment_rings       27
                   dropped_17_fragment_rings       24

## 3. Class policy: `Healthy` becomes an image-level label

A healthy leaf is not an instance of a lesion class. Keeping it as a detection category inflates mAP
and creates a category that absorbs every leaf-like region, which is the direct cause of the
out-of-domain behaviour ("any image returns a disease"). Healthy images are kept as **background
images with no labels**, which is exactly how Ultralytics learns to emit nothing.

In [4]:
for domain in DOMAINS:
    coco = SOURCE[domain]["coco"]
    id_to_name = {int(c["id"]): str(c["name"]) for c in coco["categories"]}
    detect = DETECTION_CLASSES[domain]
    instance_anns, demoted = [], 0
    for ann in SOURCE[domain]["repaired_annotations"]:
        name = id_to_name[int(ann["category_id"])]
        if name in IMAGE_LEVEL_LABELS:
            demoted += 1
            continue
        if name not in detect:
            continue
        ann = dict(ann)
        ann["category_id"] = detect.index(name)
        instance_anns.append(ann)
    SOURCE[domain]["instance_annotations"] = instance_anns
    SOURCE[domain]["id_to_name"] = id_to_name
    print(f"{domain:7s} classes={detect} instances={len(instance_anns)} demoted_to_image_level={demoted}")

rice    classes=['BrownSpot', 'Hispa', 'LeafBlast'] instances=1851 demoted_to_image_level=1425
coffee  classes=['LeafMiner', 'PowderyMildew', 'Rust', 'AlgalLeafSpot'] instances=3716 demoted_to_image_level=0


## 3b. Image usability filter

An image whose only annotations were dropped is **not** a negative: it still shows the disease named
by its folder label, so training on it as background teaches false negatives. Only images whose
label is image-level (`Healthy`) become legitimate background images; diseased images that lost
every instance are excluded and logged.

In [5]:
for domain in DOMAINS:
    images = SOURCE[domain]["images"].copy()
    per_image = defaultdict(int)
    for ann in SOURCE[domain]["instance_annotations"]:
        per_image[int(ann["image_id"])] += 1
    images["instance_count"] = images["coco_image_id"].astype(int).map(per_image).fillna(0).astype(int)
    is_image_level = images["label"].isin(IMAGE_LEVEL_LABELS)
    keep = (images["instance_count"] > 0) | is_image_level

    excluded = images.loc[~keep, ["sample_id", "label", "coco_file_name", "instance_count"]].copy()
    excluded["reason"] = "diseased_image_lost_all_instances"
    SOURCE[domain]["excluded_images"] = excluded
    SOURCE[domain]["images"] = images.loc[keep].reset_index(drop=True)

    negatives = int((SOURCE[domain]["images"]["instance_count"] == 0).sum())
    print(f"{domain:7s} usable={len(SOURCE[domain]['images'])} excluded={len(excluded)} "
          f"background_images={negatives}")
    if len(excluded):
        print(excluded["label"].value_counts().to_string())

rice    usable=3068 excluded=283 background_images=1472
label
LeafBlast    176
BrownSpot     73
Hispa         34
coffee  usable=3716 excluded=82 background_images=0
label
AlgalLeafSpot    29
Rust             28
LeafMiner        22
PowderyMildew     3


## 4. Grouping

Split units are **groups**, never single images.

* Coffee: group = source photo id (`IMG_<digits>`). Only one canonical variant per group is kept; the
  other two are flip/rot copies (mean abs pixel diff 0.21/255 after flip/rot alignment, vs 56.0 for
  random pairs) and belong to online augmentation, not to the dataset.
* Rice: group = capture burst (filename timestamps within `BURST_GAP_SECONDS`) unioned with verified
  near-duplicate pairs. Global image similarity alone is unusable here: the studio background is
  uniform, so distance-based clustering chains into one giant component (measured: a single
  component of 2522 images at a tolerant threshold). Timestamp bursts are the reliable signal, and
  the pixel tolerance is kept strict (`DUP_PIXEL_TOL`) to stay precision-first.

In [6]:
def phash_bits(path: Path, size: int = 32) -> np.ndarray:
    arr = np.asarray(Image.open(path).convert("L").resize((size, size)), dtype=np.float32)
    return (arr - arr.mean() > 0).ravel()


def thumbnail(path: Path, size: int = 64) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L").resize((size, size)), dtype=np.float32)


def flip_invariant_diff(a: np.ndarray, b: np.ndarray) -> float:
    return float(min(np.abs(a - c).mean() for c in (b, np.fliplr(b), np.flipud(b), np.rot90(b, 2))))


def duplicate_pairs(paths: dict[str, Path]) -> list[tuple[str, str, float]]:
    """pHash pre-filter, then flip/rot-invariant pixel verification."""
    if not ENABLE_DUP_SCAN or len(paths) < 2:
        return []
    ids = list(paths)
    bits = np.stack([phash_bits(paths[i]) for i in ids]).astype(np.float32)
    n_bits = bits.shape[1]
    hamming = n_bits - (bits @ bits.T + (1.0 - bits) @ (1.0 - bits).T)
    np.fill_diagonal(hamming, n_bits)
    thumbs = {i: thumbnail(paths[i]) for i in ids}
    pairs = []
    for i, j in np.argwhere(hamming < PHASH_CANDIDATE_BITS):
        if i >= j:
            continue
        diff = flip_invariant_diff(thumbs[ids[i]], thumbs[ids[j]])
        if diff < DUP_PIXEL_TOL:
            pairs.append((ids[i], ids[j], round(diff, 3)))
    return pairs


class UnionFind:
    def __init__(self, items):
        self.parent = {item: item for item in items}

    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra


TIMESTAMP_RE = re.compile(r"(\d{8})[_-](\d{6})")
SOURCE_ID_RE = re.compile(r"^(IMG[_-]?\d+)")


def coffee_groups(images: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    out = images.copy()
    out["source_id"] = out["file_name"].astype(str).str.extract(SOURCE_ID_RE)[0]
    out["source_id"] = out["source_id"].fillna(out["sample_id"])
    out["group_id"] = "src:" + out["source_id"].astype(str)
    out["group_reason"] = "source_photo_id"
    out = out.sort_values("sample_id")                       # deterministic canonical variant
    out["variant_rank"] = out.groupby("group_id").cumcount()
    out["is_canonical"] = out["variant_rank"] == 0
    return out, []


def rice_groups(images: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    out = images.copy()
    stamps = out["file_name"].astype(str).str.extract(TIMESTAMP_RE)
    out["capture_session"] = stamps[0]
    out["capture_ts"] = pd.to_datetime(stamps[0].fillna("") + stamps[1].fillna(""),
                                       format="%Y%m%d%H%M%S", errors="coerce")
    finder = UnionFind(out["sample_id"].tolist())
    timed = out.dropna(subset=["capture_ts"]).sort_values("capture_ts")
    gaps = timed["capture_ts"].diff().dt.total_seconds().fillna(np.inf)
    bursts = (gaps > BURST_GAP_SECONDS).cumsum()
    for _, part in timed.assign(_burst=bursts.values).groupby("_burst"):
        members = part["sample_id"].tolist()
        for other in members[1:]:
            finder.union(members[0], other)
    paths = {row.sample_id: resolve_image_path("rice", row.coco_file_name) for row in out.itertuples()}
    pairs = duplicate_pairs(paths)
    for a, b, _ in pairs:
        finder.union(a, b)
    out["group_id"] = ["grp:" + finder.find(s) for s in out["sample_id"]]
    out["group_reason"] = np.where(out["capture_ts"].notna(), "capture_burst", "singleton")
    out["variant_rank"] = 0
    out["is_canonical"] = True
    return out, pairs


GROUPED = {}
for domain in DOMAINS:
    started = time.time()
    builder = coffee_groups if domain == "coffee" else rice_groups
    grouped, dup_pairs = builder(SOURCE[domain]["images"])
    GROUPED[domain] = {"images": grouped, "dup_pairs": dup_pairs}
    sizes = grouped.groupby("group_id").size()
    print(f"{domain:7s} images={len(grouped)} groups={sizes.size} max_group={sizes.max()} "
          f"canonical={int(grouped['is_canonical'].sum())} verified_dup_pairs={len(dup_pairs)} "
          f"({time.time() - started:.0f}s)")

rice    images=3068 groups=660 max_group=60 canonical=3068 verified_dup_pairs=27 (243s)
coffee  images=3716 groups=1256 max_group=3 canonical=1256 verified_dup_pairs=0 (0s)


## 5. Grouped stratified split

Groups are assigned largest-first to the split with the largest remaining deficit for the group's
majority class. Class proportions stay close to 70/15/15 and no group is ever split.

In [7]:
def grouped_stratified_split(images: pd.DataFrame, ratio: dict[str, float]) -> pd.DataFrame:
    frame = images[images["is_canonical"]].copy()
    groups = (frame.groupby("group_id")
                   .agg(size=("sample_id", "size"),
                        majority=("label", lambda s: s.value_counts().idxmax()))
                   .reset_index())
    targets = {label: {split: part["size"].sum() * share for split, share in ratio.items()}
               for label, part in groups.groupby("majority")}
    filled = {label: {split: 0.0 for split in ratio} for label in targets}
    assignment = {}
    rng = np.random.default_rng(SEED)
    order = groups.sample(frac=1.0, random_state=SEED).sort_values("size", ascending=False, kind="stable")
    for row in order.itertuples():
        deficits = {split: targets[row.majority][split] - filled[row.majority][split] for split in ratio}
        best = max(deficits, key=lambda split: (deficits[split], rng.random()))
        assignment[row.group_id] = best
        filled[row.majority][best] += row.size
    out = images.copy()
    out["split"] = out["group_id"].map(assignment)
    out.loc[~out["is_canonical"], "split"] = "excluded_duplicate_variant"
    return out


for domain in DOMAINS:
    grouped = grouped_stratified_split(GROUPED[domain]["images"], SPLIT_RATIO)
    GROUPED[domain]["images"] = grouped
    active = grouped[grouped["split"].isin(SPLIT_RATIO)]
    print(f"=== {domain}: active={len(active)} excluded_variants={len(grouped) - len(active)}")
    table = pd.crosstab(active["label"], active["split"])
    print(table.to_string())
    print((table / table.sum()).round(3).to_string())

=== rice: active=3068 excluded_variants=0


split      test  train  val
label                      
BrownSpot    71    319   68
Healthy     208   1036  228
Hispa        90    379   68
LeafBlast    92    414   95
split       test  train    val
label                         
BrownSpot  0.154  0.149  0.148
Healthy    0.451  0.482  0.497
Hispa      0.195  0.176  0.148
LeafBlast  0.200  0.193  0.207
=== coffee: active=1256 excluded_variants=2460
split          test  train  val
label                          
AlgalLeafSpot    46    216   47
LeafMiner        59    277   60
PowderyMildew    13     61   13
Rust             69    325   70
split           test  train    val
label                             
AlgalLeafSpot  0.246  0.246  0.247
LeafMiner      0.316  0.315  0.316
PowderyMildew  0.070  0.069  0.068
Rust           0.369  0.370  0.368


## 6. Leakage and shortcut audit

Hard assertions: no group spans two splits, no md5 repeats across splits, and no verified
flip/rot duplicate pair crosses splits. The capture-session table is printed so the session
shortcut stays visible instead of silent.

In [8]:
AUDIT = {}
for domain in DOMAINS:
    grouped = GROUPED[domain]["images"]
    active = grouped[grouped["split"].isin(SPLIT_RATIO)].copy()

    crossing = active.groupby("group_id")["split"].nunique()
    assert int((crossing > 1).sum()) == 0, f"{domain}: a group spans multiple splits"

    md5_split = active.groupby("md5")["split"].nunique()
    assert int((md5_split > 1).sum()) == 0, f"{domain}: identical md5 across splits"

    split_of = dict(zip(active["sample_id"], active["split"]))
    if domain == "coffee":
        pairs = [(a, b, 0.0)
                 for _, part in active.groupby("source_id")
                 for a in part["sample_id"] for b in part["sample_id"] if a < b]
    else:
        pairs = GROUPED[domain]["dup_pairs"]
    crossing_pairs = [(a, b) for a, b, *_ in pairs
                      if a in split_of and b in split_of and split_of[a] != split_of[b]]
    assert not crossing_pairs, f"{domain}: {len(crossing_pairs)} duplicate pairs cross splits"

    session_col = "source_id" if domain == "coffee" else "capture_session"
    session_table = pd.crosstab(active[session_col].fillna("unknown"), active["label"])

    AUDIT[domain] = {
        "images_active": int(len(active)),
        "images_excluded_duplicate_variants": int(len(grouped) - len(active)),
        "groups": int(active["group_id"].nunique()),
        "groups_crossing_splits": 0,
        "md5_collisions_across_splits": 0,
        "duplicate_pairs_checked": int(len(pairs)),
        "duplicate_pairs_crossing_splits": 0,
        "distinct_capture_sessions": int(active[session_col].nunique(dropna=True)),
        "split_counts": active["split"].value_counts().to_dict(),
        "class_split_counts": pd.crosstab(active["label"], active["split"]).to_dict(),
    }
    print(f"=== {domain} audit passed")
    print(json.dumps({k: v for k, v in AUDIT[domain].items() if k != "class_split_counts"}, indent=2))
    if domain == "rice":
        print(session_table.to_string())

=== rice audit passed
{
  "images_active": 3068,
  "images_excluded_duplicate_variants": 0,
  "groups": 660,
  "groups_crossing_splits": 0,
  "md5_collisions_across_splits": 0,
  "duplicate_pairs_checked": 27,
  "duplicate_pairs_crossing_splits": 0,
  "distinct_capture_sessions": 4,
  "split_counts": {
    "train": 2148,
    "test": 461,
    "val": 459
  }
}
label            BrownSpot  Healthy  Hispa  LeafBlast
capture_session                                      
20190419                54      952    438        481
20190420               293       67     32         38
20190421                33       45      0          4
20190424                33      362     53          6
unknown                 45       46     14         72
=== coffee audit passed
{
  "images_active": 1256,
  "images_excluded_duplicate_variants": 2460,
  "groups": 1256,
  "groups_crossing_splits": 0,
  "md5_collisions_across_splits": 0,
  "duplicate_pairs_checked": 0,
  "duplicate_pairs_crossing_splits": 0,
  "dis

## 7. Write `coffee_rice_v002`

In [9]:

def populate_image(source: Path, target: Path, mode: str = "auto") -> None:
    if target.exists():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    resolved = mode
    if resolved == "auto":
        resolved = "clone" if platform.system() == "Darwin" else "symlink"
    if resolved == "clone":
        try:
            subprocess.run(["cp", "-c", str(source), str(target)], check=True, capture_output=True)
            return
        except (subprocess.CalledProcessError, FileNotFoundError):
            resolved = "symlink"
    if resolved == "symlink":
        try:
            target.symlink_to(source.resolve())
            return
        except OSError:
            pass
    shutil.copy2(source, target)

def build_coco(domain: str, active: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    detect = DETECTION_CLASSES[domain]
    keep_ids = set(active["coco_image_id"].astype(int))
    by_image = defaultdict(list)
    for ann in SOURCE[domain]["instance_annotations"]:
        if int(ann["image_id"]) in keep_ids:
            by_image[int(ann["image_id"])].append(ann)

    images, annotations, rows = [], [], []
    next_id = 1
    for row in active.itertuples():
        image_id = int(row.coco_image_id)
        anns = by_image.get(image_id, [])
        images.append({"id": image_id, "file_name": row.coco_file_name,
                       "width": int(row.width), "height": int(row.height),
                       "split": row.split, "image_label": row.label, "group_id": row.group_id})
        for ann in anns:
            out = dict(ann); out["id"] = next_id; next_id += 1
            annotations.append(out)
        rows.append({
            "sample_id": row.sample_id, "split": row.split, "domain": domain,
            "image_label": row.label, "is_negative": int(not anns), "instance_count": len(anns),
            "coco_image_id": image_id, "coco_file_name": row.coco_file_name,
            "source_relpath": row.clean_relpath, "width": int(row.width), "height": int(row.height),
            "md5": row.md5, "group_id": row.group_id, "group_reason": row.group_reason,
            "dataset_version": TARGET_VERSION,
        })
    coco = {
        "info": {"description": f"{domain} {TARGET_VERSION} repaired instance segmentation",
                 "source_version": SOURCE_VERSION, "seed": SEED},
        "licenses": [],
        "categories": [{"id": i, "name": name, "supercategory": domain} for i, name in enumerate(detect)],
        "images": images,
        "annotations": annotations,
    }
    return coco, pd.DataFrame(rows)


MANIFEST = {}
for domain in DOMAINS:
    grouped = GROUPED[domain]["images"]
    active = grouped[grouped["split"].isin(SPLIT_RATIO)]
    coco, manifest = build_coco(domain, active)
    MANIFEST[domain] = manifest

    domain_dir = OUTPUT_ROOT / domain
    (domain_dir / "annotations").mkdir(parents=True, exist_ok=True)
    (domain_dir / "manifests").mkdir(parents=True, exist_ok=True)
    with (domain_dir / "annotations" / "instances.coco.json").open("w", encoding="utf-8") as handle:
        json.dump(coco, handle)
    manifest.to_csv(domain_dir / "manifests" / "images.csv", index=False)
    for split in SPLIT_RATIO:
        manifest[manifest["split"] == split].to_csv(domain_dir / "manifests" / f"{split}.csv", index=False)
    (domain_dir / "class_mapping.json").write_text(json.dumps({
        "domain": domain,
        "dataset_version": TARGET_VERSION,
        "detection_classes": DETECTION_CLASSES[domain],
        "class_to_id": {name: i for i, name in enumerate(DETECTION_CLASSES[domain])},
        "image_level_labels": sorted(IMAGE_LEVEL_LABELS),
    }, indent=2), encoding="utf-8")

    SOURCE[domain]["ledger"].to_csv(REPORT_DIR / f"{domain}_annotation_ledger.csv", index=False)
    SOURCE[domain]["excluded_images"].to_csv(REPORT_DIR / f"{domain}_excluded_images.csv", index=False)
    grouped[["sample_id", "group_id", "group_reason", "split", "label", "is_canonical"]].to_csv(
        REPORT_DIR / f"{domain}_groups.csv", index=False)
    if GROUPED[domain]["dup_pairs"]:
        pd.DataFrame(GROUPED[domain]["dup_pairs"], columns=["a", "b", "pixel_diff"]).to_csv(
            REPORT_DIR / f"{domain}_duplicate_pairs.csv", index=False)
    manifest[manifest["instance_count"] > 0].to_csv(domain_dir / "manifests" / "segmentation.csv", index=False)

    if POPULATE_IMAGES:
        for row in active.itertuples():
            src_path = resolve_image_path(domain, row.coco_file_name)
            dst_path = domain_dir / row.coco_file_name
            populate_image(src_path, dst_path, mode=IMAGE_LINK_MODE)

    print(f"{domain:7s} wrote images={len(coco['images'])} instances={len(coco['annotations'])} "
          f"negatives={int(manifest['is_negative'].sum())}")

# Write top-level taxonomy.json and metadata/
taxonomy = {
    "dataset_version": TARGET_VERSION,
    "domains": {
        domain: {
            "detection_classes": DETECTION_CLASSES[domain],
            "class_to_id": {name: i for i, name in enumerate(DETECTION_CLASSES[domain])},
            "image_level_labels": sorted(IMAGE_LEVEL_LABELS & set(SOURCE[domain]["images"]["label"].unique())),
        }
        for domain in DOMAINS
    },
}
(OUTPUT_ROOT / "taxonomy.json").write_text(json.dumps(taxonomy, indent=2), encoding="utf-8")

metadata_dir = OUTPUT_ROOT / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)
all_manifests = pd.concat([MANIFEST[d] for d in DOMAINS], ignore_index=True)
all_manifests.to_csv(metadata_dir / "all_splits_manifest.csv", index=False)
for split in SPLIT_RATIO:
    all_manifests[all_manifests["split"] == split].to_csv(metadata_dir / f"{split}_manifest.csv", index=False)


rice    wrote images=3068 instances=1851 negatives=1472


coffee  wrote images=1256 instances=1256 negatives=0


In [10]:
repair_config = {
    "dataset_version": TARGET_VERSION,
    "source_version": SOURCE_VERSION,
    "images_source_version": SOURCE_VERSION,
    "images_copied": POPULATE_IMAGES,
    "images_link_mode": IMAGE_LINK_MODE,
    "random_seed": SEED,
    "split_ratio": SPLIT_RATIO,
    "split_strategy": {"rice": "grouped_by_capture_burst_and_verified_duplicates",
                       "coffee": "grouped_by_source_photo_id_one_canonical_variant"},
    "instance_policy": {"one_instance_per_annotation": True, "ring_selection": "largest_ring",
                        "min_ring_points": MIN_RING_POINTS, "min_area_frac": MIN_AREA_FRAC,
                        "max_area_frac": MAX_AREA_FRAC},
    "class_policy": {"image_level_labels": sorted(IMAGE_LEVEL_LABELS),
                     "detection_classes": DETECTION_CLASSES,
                     "image_level_images_become_negatives": True},
    "duplicate_detection": {"enabled": ENABLE_DUP_SCAN, "phash_bits": 1024,
                            "phash_candidate_hamming": PHASH_CANDIDATE_BITS,
                            "pixel_tolerance": DUP_PIXEL_TOL, "flip_rot_invariant": True},
    "burst_gap_seconds": BURST_GAP_SECONDS,
}

dataset_manifest = {
    "dataset_version": TARGET_VERSION,
    "images_source_version": SOURCE_VERSION,
    "domains": {
        domain: {
            "detection_classes": DETECTION_CLASSES[domain],
            "n_images": int(len(MANIFEST[domain])),
            "n_negative_images": int(MANIFEST[domain]["is_negative"].sum()),
            "n_instances": int(MANIFEST[domain]["instance_count"].sum()),
            "instances_per_positive_image": round(float(
                MANIFEST[domain].loc[MANIFEST[domain]["instance_count"] > 0, "instance_count"].mean()), 3),
            "splits": MANIFEST[domain]["split"].value_counts().to_dict(),
        }
        for domain in DOMAINS
    },
}

(OUTPUT_ROOT / "repair_config.json").write_text(json.dumps(repair_config, indent=2), encoding="utf-8")
(OUTPUT_ROOT / "metadata" / "preprocessing_config.json").write_text(json.dumps(repair_config, indent=2), encoding="utf-8")
(OUTPUT_ROOT / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")
(REPORT_DIR / "leakage_audit.json").write_text(json.dumps(AUDIT, indent=2, default=str), encoding="utf-8")

before_after = pd.DataFrame([
    {
        "domain": domain,
        "v001_images": SOURCE[domain]["n_images_source"],
        "v001_annotations": int(len(SOURCE[domain]["coco"]["annotations"])),
        "v001_yolo_instances_per_ring_export": int(sum(
            1 for ann in SOURCE[domain]["coco"]["annotations"]
            for ring in (ann.get("segmentation") or [])
            if isinstance(ring, list) and len(ring) >= 6)),
        "v002_images": int(len(MANIFEST[domain])),
        "v002_instances": int(MANIFEST[domain]["instance_count"].sum()),
        "v002_negatives": int(MANIFEST[domain]["is_negative"].sum()),
        "dropped_annotations": int((SOURCE[domain]["ledger"]["action"] == "dropped").sum()),
        "excluded_images": int(len(SOURCE[domain]["excluded_images"])),
    }
    for domain in DOMAINS
])
before_after.to_csv(REPORT_DIR / "v001_vs_v002.csv", index=False)
print(json.dumps(dataset_manifest, indent=2))
display(before_after)
print(sorted(p.name for p in REPORT_DIR.iterdir()))

{
  "dataset_version": "coffee_rice_v002",
  "images_source_version": "coffee_rice_v001",
  "domains": {
    "rice": {
      "detection_classes": [
        "BrownSpot",
        "Hispa",
        "LeafBlast"
      ],
      "n_images": 3068,
      "n_negative_images": 1472,
      "n_instances": 1851,
      "instances_per_positive_image": 1.16,
      "splits": {
        "train": 2148,
        "test": 461,
        "val": 459
      }
    },
    "coffee": {
      "detection_classes": [
        "LeafMiner",
        "PowderyMildew",
        "Rust",
        "AlgalLeafSpot"
      ],
      "n_images": 1256,
      "n_negative_images": 0,
      "n_instances": 1256,
      "instances_per_positive_image": 1.0,
      "splits": {
        "train": 879,
        "val": 190,
        "test": 187
      }
    }
  }
}


,domain,v001_images,v001_annotations,v001_yolo_instances_per_ring_export,v002_images,v002_instances,v002_negatives,dropped_annotations,excluded_images
0,rice,3351,3646,3715,3068,1851,1472,370,283
1,coffee,3798,3774,30791,1256,1256,0,58,82


['coffee_annotation_ledger.csv', 'coffee_excluded_images.csv', 'coffee_groups.csv', 'leakage_audit.json', 'rice_annotation_ledger.csv', 'rice_duplicate_pairs.csv', 'rice_excluded_images.csv', 'rice_groups.csv', 'v001_vs_v002.csv']
